# Предобработка корпуса текстов

Цель — подготовить авторские корпуса для обучения Word2Vec: **очистка → предложения → лемматизация + POS → фильтрация стоп-слов**.

Как в практической части статьи, единица анализа — токен вида `lemma_POS`: это помогает **в большинстве случаев устранить омонимию** и сделать сравнение моделей более интерпретируемым.

## Входные данные
- `data/raw/*.zip` — **1 zip = 1 автор** (внутри: `.txt` и/или `.xml`)

## Выходные данные
- `data/lemmas_pos_sentences/<author>.txt` — **1 строка = 1 предложение**, токены `lemma_POS` разделены пробелами.

## Что именно делает пайплайн
- распаковывает архивы по авторам
- собирает тексты в один файл на автора (XML → текст через `BeautifulSoup`)
- очищает от мусорных символов
- сегментирует на предложения
- лемматизирует и добавляет POS-теги (`pymystem3`)
- удаляет русские стоп-слова (`nltk.stopwords`)


In [ ]:
import re
import zipfile
from pathlib import Path

from bs4 import BeautifulSoup
from joblib import Parallel, delayed
from pymystem3 import Mystem
from tqdm import tqdm

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

BASE = Path.cwd()
RAW_DIR = BASE / "data" / "raw"
COLLECTED_DIR = BASE / "data" / "collected"
OUT_DIR = BASE / "data" / "lemmas_pos_sentences"

COLLECTED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

STOP = set(stopwords.words("russian"))

N_JOBS = 4
BATCH_SIZE = 500


In [ ]:
# 1) Распаковка архивов: каждый zip — в папку с именем автора
for zpath in RAW_DIR.glob("*.zip"):
    author = zpath.stem
    out_dir = RAW_DIR / author
    out_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(out_dir)
    print(author, "— распакован в", out_dir)


In [ ]:
# 2) Сбор текстов: XML → текст через bs4, txt → как есть

def read_file_text(path: Path) -> str:
    raw = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix.lower() == ".xml":
        soup = BeautifulSoup(raw, "xml")
        return soup.get_text(separator=" ", strip=True)
    return raw


for author_dir in RAW_DIR.iterdir():
    if not author_dir.is_dir():
        continue

    parts: list[str] = []
    for path in author_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in (".txt", ".xml"):
            parts.append(read_file_text(path))

    if not parts:
        continue

    out_file = COLLECTED_DIR / f"{author_dir.name}.txt"
    out_file.write_text("\n\n".join(parts), encoding="utf-8")
    print(author_dir.name, "— собран в", out_file)


In [ ]:
# 3) Очистка → предложения → леммы_POS

SEP = "<|>"


def clean_text(text: str) -> str:
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s.!?]", " ", text, flags=re.U)
    return text


def split_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def to_pos(gr: str) -> str:
    return gr.split("=")[0].split(",")[0].strip() if gr else "UNK"


def lemmatize_batch_sentences(sentences: list[str]) -> list[list[str]]:
    m = Mystem()
    merged = SEP.join(sentences)

    out: list[list[str]] = []
    cur: list[str] = []

    for item in m.analyze(merged):
        if item.get("text") == SEP:
            if cur:
                out.append(cur)
            cur = []
            continue

        a = (item.get("analysis") or [None])[0]
        if not a:
            continue

        lemma = (a.get("lex") or "").strip().lower()
        if not lemma or lemma in STOP:
            continue

        cur.append(f"{lemma}_{to_pos(a.get('gr') or '')}")

    if cur:
        out.append(cur)

    return out


def chunked(items: list[str], size: int) -> list[list[str]]:
    return [items[i : i + size] for i in range(0, len(items), size)]


In [ ]:
# 4) По каждому автору: батчи предложений → последовательно, а внутри батча – параллельно → 1 строка = 1 предложение

for txt_path in COLLECTED_DIR.glob("*.txt"):
    author = txt_path.stem
    text = clean_text(txt_path.read_text(encoding="utf-8"))
    sentences = split_sentences(text)
    batches = chunked(sentences, BATCH_SIZE)
    total_batches = len(batches)
    print(f"{author}: всего батчей {total_batches}")

    out_path = OUT_DIR / f"{author}.txt"
    with out_path.open("w", encoding="utf-8") as f:
        for batch in tqdm(batches, desc=f"{author}", unit="batch"):
            processed = lemmatize_batch_sentences(batch)
            for toks in processed:
                if toks:
                    f.write(" ".join(toks) + "\n")
       

